In [1]:
from dotenv import load_dotenv
load_dotenv("../Lesson1/.env")


True

In [2]:
from openai import OpenAI
import os

# Initialize client using the variables injected from your .env
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

print(os.getenv("BASE_URL"))

https://models.inference.ai.azure.com


In [3]:
from ingest import load_faq_data
raw_documents = load_faq_data()

texts = [doc["filename"] + " " + doc["content"] for doc in raw_documents]


In [4]:
from tqdm.auto import tqdm
from embedder import Embedder
import numpy as np


embed = Embedder()

batch_size = 50
embeddings = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    embeddings.extend(batch_vectors)

embeddings = np.array(embeddings)

2026-06-21 04:45:06.192877480 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


  0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
Q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(Q1)


In [6]:
print (f'Answer Q1 - {v1[0]}')

Answer Q1 - -0.02058203437252893


In [7]:
target_doc = next(doc for doc in raw_documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

In [8]:
v2 = embed.encode(target_doc["content"])

In [9]:
print (f'Answer Q2 - {round(v1.dot(v2), 2)}')

Answer Q2 - 0.36


In [10]:
from ingest import build_chunks
 
chunks = build_chunks(raw_documents)


In [11]:
batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_texts = [chunk["content"] for chunk in batch]
    batch_vectors = embed.encode_batch(batch_texts)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [12]:
scores = X.dot(v1)
max_idx = np.argmax(scores)
print(f'Answer Q3: {chunks[max_idx]["filename"]}')

Answer Q3: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [13]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(X, chunks)

In [14]:
q4 = "What metric do we use to evaluate a search engine?"
v4 = embed.encode(q4)

In [15]:
results = vindex.search(query_vector=v4, num_results=5)
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [ ]:
q5 ="How do I store vectors in PostgreSQL?"
# %% 
# Replace your Q5 text index block with this setup:
import minsearch

# Manually build the text index using the fields present in your chunks
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)


text_results = index.search(
    query=q5,
    filter_dict={},
    boost_dict={"content": 1.0},
    num_results=5
)

print("--- TEXT SEARCH FILENAMES ---")
for doc in text_results:
    print(doc["filename"])


# %%
# 2. Run the vector search 
q5_vector = embed.encode(q5)
vector_results = vindex.search(
    query_vector=q5_vector,
    num_results=5
)

print("\n--- VECTOR SEARCH FILENAMES ---")
for doc in vector_results:
    print(doc["filename"])

--- TEXT SEARCH FILENAMES ---
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

--- VECTOR SEARCH FILENAMES ---
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [17]:
# 2. Run the vector search
q5_vector = embed.encode(q5)
vector_results = vindex.search(
    query_vector=q5_vector,
    num_results=5
)

print("\n--- VECTOR SEARCH FILENAMES ---")
for doc in vector_results:
    print(doc["filename"])


--- VECTOR SEARCH FILENAMES ---
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [19]:
q6 ="How do I give the model access to tools?"

In [20]:
text_results_N = index.search(
    query=q6,
    filter_dict={},
    boost_dict={"content": 1.0},
    num_results=5
)

In [21]:
q6_vector = embed.encode(q6)
vector_results_N = vindex.search(
    query_vector=q6_vector,
    num_results=5
)

text_results_N.extend(vector_results_N)

In [22]:
print(text_results_N)

[{'start': 0, 'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can c

In [23]:
results = rrf([vector_results, text_results_N])

In [25]:
for doc in results:
    print(doc["filename"])

01-agentic-rag/lessons/13-function-calling.md
02-vector-search/lessons/08-pgvector.md
01-agentic-rag/lessons/14-agentic-loop.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
